# EfficientAT MN10 Drone Audio Classifier — Colab Training

Train a binary drone/background classifier on the DADS dataset using EfficientAT MobileNetV3 MN10 pretrained on AudioSet.

## Preprocessing Compatibility Notice

**CRITICAL**: This notebook uses `AugmentMelSTFT` — the *exact* same preprocessing class as the local inference code in `src/acoustic/classification/efficientat/preprocess.py`. Every parameter (n_fft, hop_length, win_length, mel banks, log normalization) is identical to what runs on device. Do NOT substitute `torchaudio.transforms.MelSpectrogram` or any other mel pipeline — it will produce a different feature distribution and the trained weights will be incompatible with local inference.

### Key preprocessing parameters
| Parameter | Value |
|-----------|-------|
| Sample rate | 32000 Hz |
| n_fft | 1024 |
| hop_length | 320 |
| win_length | 800 |
| window | Hann (periodic=False) |
| n_mels | 128 |
| fmin | 0 Hz |
| fmax | 15000 Hz |
| normalization | (log(mel + 1e-5) + 4.5) / 5.0 |
| SpecAugment (train) | FreqMask=48, TimeMask=192 |
| SpecAugment (eval) | disabled |

## Cell 2: Install dependencies

In [ ]:
!pip install -q torch torchaudio torchvision datasets pyarrow soundfile huggingface_hub

# Authenticate with HuggingFace for faster downloads (uses cached token or prompts login)
from huggingface_hub import login
import os

# Set your HF token here for non-interactive use, or leave empty to use notebook_login()
HF_TOKEN = ""  # paste your token here, or set HF_TOKEN env var

token = HF_TOKEN or os.environ.get("HF_TOKEN", "")
if token:
    login(token=token)
    print("Logged in to HuggingFace with token")
else:
    from huggingface_hub import notebook_login
    notebook_login()


## Cell 3: Configuration

All hyperparameters in one place. Modify here only.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Audio
SAMPLE_RATE = 32000          # target sample rate (resample 16kHz → 32kHz)
SEGMENT_SAMPLES = 32000      # 1-second segments
SOURCE_SAMPLE_RATE = 16000   # DADS dataset native rate

# Mel spectrogram (MUST match local inference AugmentMelSTFT exactly)
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 320
WIN_LENGTH = 800
FMIN = 0.0
FMAX = 15000.0               # sr//2 - 1000 = 32000//2 - 1000
FREQM_TRAIN = 48             # frequency masking during training
TIMEM_TRAIN = 192            # time masking during training

# Model
NUM_CLASSES_PRETRAINED = 527  # AudioSet classes in pretrained weights
WIDTH_MULT = 1.0
INPUT_DIM_F = 128             # mel bands
INPUT_DIM_T = 100             # time frames (~1s at 32kHz with hop=320)

# Training stages
STAGE1_EPOCHS = 10   # head only, lr=1e-3
STAGE2_EPOCHS = 15   # head + last 3 feature blocks, lr=1e-4
STAGE3_EPOCHS = 20   # all layers, lr=1e-5
EARLY_STOP_PATIENCE = 5

# Stage learning rates
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-4
LR_STAGE3 = 1e-5

# Focal loss
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

# DataLoader
BATCH_SIZE = 32
NUM_WORKERS = 2

# Reproducibility
SEED = 42

# Paths
PRETRAINED_URL = "https://github.com/fschmid56/EfficientAT/releases/download/v0.0.1/mn10_as_mAP_471.pt"
PRETRAINED_PATH = "mn10_as.pt"
CHECKPOINT_PATH = "efficientat_mn10_local.pt"

# HuggingFace dataset
HF_DATASET = "geronimobasso/drone-audio-detection-samples"

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 4: Download pretrained weights

Downloads `mn10_as_mAP_471.pt` from EfficientAT GitHub releases.

In [ ]:
import os
import subprocess

if not os.path.exists(PRETRAINED_PATH):
    print(f"Downloading pretrained weights...")
    # Use wget for reliable large file download from GitHub releases
    result = subprocess.run(
        ["wget", "-q", "--show-progress", "-O", PRETRAINED_PATH, PRETRAINED_URL],
        capture_output=False,
    )
    if result.returncode != 0:
        # Fallback to urllib
        import urllib.request
        print("wget failed, trying urllib...")
        urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)
    print(f"Saved to: {PRETRAINED_PATH}")
else:
    print(f"Pretrained weights already present: {PRETRAINED_PATH}")

size_mb = os.path.getsize(PRETRAINED_PATH) / 1e6
print(f"File size: {size_mb:.1f} MB")


## Cell 5: Model code (vendored from EfficientAT)

Exact copy of the local model source — `MN`, `get_model`, `InvertedResidual`, `InvertedResidualConfig`, `ConcurrentSEBlock`, `SqueezeExcitation`, `MultiHeadAttentionPooling`, and utilities. No modifications.

In [ ]:
# ── utils.py ──────────────────────────────────────────────────────────────────
import math
from functools import partial
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torchvision.ops.misc import ConvNormActivation


def make_divisible(v: float, divisor: int, min_value: Optional[int] = None) -> int:
    """Ensure channel count is divisible by 8 (from TF MobileNet repo)."""
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


def cnn_out_size(in_size: int, padding: int, dilation: int, kernel: int, stride: int) -> int:
    """Compute output size of a CNN layer."""
    s = in_size + 2 * padding - dilation * (kernel - 1) - 1
    return math.floor(s / stride + 1)


def collapse_dim(
    x: Tensor,
    dim: int,
    mode: str = "pool",
    pool_fn: Callable[[Tensor, int], Tensor] = torch.mean,
    combine_dim: Optional[int] = None,
) -> Tensor:
    """Collapse a dimension by pooling or combining."""
    if mode == "pool":
        return pool_fn(x, dim)
    elif mode == "combine":
        s = list(x.size())
        s[combine_dim] *= dim
        s[dim] //= dim
        return x.view(s)
    raise ValueError(f"Unknown mode: {mode}")


# ── attention_pooling.py ──────────────────────────────────────────────────────

class MultiHeadAttentionPooling(nn.Module):
    """Multi-Head Attention as used in PSLA paper (https://arxiv.org/pdf/2102.01243.pdf)."""

    def __init__(
        self,
        in_dim: int,
        out_dim: int,
        att_activation: str = "sigmoid",
        clf_activation: str = "ident",
        num_heads: int = 4,
        epsilon: float = 1e-7,
    ) -> None:
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_heads = num_heads
        self.epsilon = epsilon
        self.att_activation = att_activation
        self.clf_activation = clf_activation
        self.subspace_proj = nn.Linear(self.in_dim, self.out_dim * 2 * self.num_heads)
        self.head_weight = nn.Parameter(
            torch.tensor([1.0 / self.num_heads] * self.num_heads).view(1, -1, 1)
        )

    def activate(self, x: Tensor, activation: str) -> Tensor:
        if activation == "linear":
            return x
        elif activation == "relu":
            return F.relu(x)
        elif activation == "sigmoid":
            return torch.sigmoid(x)
        elif activation == "softmax":
            return F.softmax(x, dim=1)
        elif activation == "ident":
            return x
        return x

    def forward(self, x: Tensor) -> Tensor:
        x = collapse_dim(x, dim=2)
        x = x.transpose(1, 2)
        b, n, c = x.shape
        x = self.subspace_proj(x).reshape(b, n, 2, self.num_heads, self.out_dim).permute(2, 0, 3, 1, 4)
        att, val = x[0], x[1]
        val = self.activate(val, self.clf_activation)
        att = self.activate(att, self.att_activation)
        att = torch.clamp(att, self.epsilon, 1.0 - self.epsilon)
        att = att / torch.sum(att, dim=2, keepdim=True)
        out = torch.sum(att * val, dim=2) * self.head_weight
        out = torch.sum(out, dim=1)
        return out


# ── inverted_residual.py ──────────────────────────────────────────────────────

class ConcurrentSEBlock(torch.nn.Module):
    """Concurrent Squeeze-and-Excitation applied across multiple dimensions."""

    def __init__(self, c_dim: int, f_dim: int, t_dim: int, se_cnf: Dict) -> None:
        super().__init__()
        dims = [c_dim, f_dim, t_dim]
        self.conc_se_layers = nn.ModuleList()
        for d in se_cnf["se_dims"]:
            input_dim = dims[d - 1]
            squeeze_dim = make_divisible(input_dim // se_cnf["se_r"], 8)
            self.conc_se_layers.append(SqueezeExcitation(input_dim, squeeze_dim, d))
        if se_cnf["se_agg"] == "max":
            self.agg_op = lambda x: torch.max(x, dim=0)[0]
        elif se_cnf["se_agg"] == "avg":
            self.agg_op = lambda x: torch.mean(x, dim=0)
        elif se_cnf["se_agg"] == "add":
            self.agg_op = lambda x: torch.sum(x, dim=0)
        elif se_cnf["se_agg"] == "min":
            self.agg_op = lambda x: torch.min(x, dim=0)[0]
        else:
            raise NotImplementedError(f"SE aggregation operation '{se_cnf['se_agg']}' not implemented")

    def forward(self, input: Tensor) -> Tensor:
        se_outs = []
        for se_layer in self.conc_se_layers:
            se_outs.append(se_layer(input))
        out = self.agg_op(torch.stack(se_outs, dim=0))
        return out


class SqueezeExcitation(torch.nn.Module):
    """Squeeze-and-Excitation block from https://arxiv.org/abs/1709.01507."""

    def __init__(
        self,
        input_dim: int,
        squeeze_dim: int,
        se_dim: int,
        activation: Callable[..., torch.nn.Module] = torch.nn.ReLU,
        scale_activation: Callable[..., torch.nn.Module] = torch.nn.Sigmoid,
    ) -> None:
        super().__init__()
        self.fc1 = torch.nn.Linear(input_dim, squeeze_dim)
        self.fc2 = torch.nn.Linear(squeeze_dim, input_dim)
        assert se_dim in [1, 2, 3]
        self.se_dim = [1, 2, 3]
        self.se_dim.remove(se_dim)
        self.activation = activation()
        self.scale_activation = scale_activation()

    def _scale(self, input: Tensor) -> Tensor:
        scale = torch.mean(input, self.se_dim, keepdim=True)
        shape = scale.size()
        scale = self.fc1(scale.squeeze(2).squeeze(2))
        scale = self.activation(scale)
        scale = self.fc2(scale)
        return self.scale_activation(scale).view(shape)

    def forward(self, input: Tensor) -> Tensor:
        scale = self._scale(input)
        return scale * input


class InvertedResidualConfig:
    """Stores information for MobileNetV3 inverted residual blocks."""

    def __init__(
        self,
        input_channels: int,
        kernel: int,
        expanded_channels: int,
        out_channels: int,
        use_se: bool,
        activation: str,
        stride: int,
        dilation: int,
        width_mult: float,
    ) -> None:
        self.input_channels = self.adjust_channels(input_channels, width_mult)
        self.kernel = kernel
        self.expanded_channels = self.adjust_channels(expanded_channels, width_mult)
        self.out_channels = self.adjust_channels(out_channels, width_mult)
        self.use_se = use_se
        self.use_hs = activation == "HS"
        self.stride = stride
        self.dilation = dilation
        self.f_dim: Optional[int] = None
        self.t_dim: Optional[int] = None

    @staticmethod
    def adjust_channels(channels: int, width_mult: float) -> int:
        return make_divisible(channels * width_mult, 8)

    def out_size(self, in_size: int) -> int:
        padding = (self.kernel - 1) // 2 * self.dilation
        return cnn_out_size(in_size, padding, self.dilation, self.kernel, self.stride)


class InvertedResidual(nn.Module):
    """MobileNetV3 inverted residual block."""

    def __init__(
        self,
        cnf: InvertedResidualConfig,
        se_cnf: Dict,
        norm_layer: Callable[..., nn.Module],
        depthwise_norm_layer: Callable[..., nn.Module],
    ) -> None:
        super().__init__()
        if not (1 <= cnf.stride <= 2):
            raise ValueError("illegal stride value")

        self.use_res_connect = cnf.stride == 1 and cnf.input_channels == cnf.out_channels

        layers: List[nn.Module] = []
        activation_layer = nn.Hardswish if cnf.use_hs else nn.ReLU

        # expand
        if cnf.expanded_channels != cnf.input_channels:
            layers.append(
                ConvNormActivation(
                    cnf.input_channels,
                    cnf.expanded_channels,
                    kernel_size=1,
                    norm_layer=norm_layer,
                    activation_layer=activation_layer,
                )
            )

        # depthwise
        stride = 1 if cnf.dilation > 1 else cnf.stride
        layers.append(
            ConvNormActivation(
                cnf.expanded_channels,
                cnf.expanded_channels,
                kernel_size=cnf.kernel,
                stride=stride,
                dilation=cnf.dilation,
                groups=cnf.expanded_channels,
                norm_layer=depthwise_norm_layer,
                activation_layer=activation_layer,
            )
        )
        if cnf.use_se and se_cnf["se_dims"] is not None:
            layers.append(ConcurrentSEBlock(cnf.expanded_channels, cnf.f_dim, cnf.t_dim, se_cnf))

        # project
        layers.append(
            ConvNormActivation(
                cnf.expanded_channels,
                cnf.out_channels,
                kernel_size=1,
                norm_layer=norm_layer,
                activation_layer=None,
            )
        )

        self.block = nn.Sequential(*layers)
        self.out_channels = cnf.out_channels
        self._is_cn = cnf.stride > 1

    def forward(self, inp: Tensor) -> Tensor:
        result = self.block(inp)
        if self.use_res_connect:
            result += inp
        return result


# ── model.py ──────────────────────────────────────────────────────────────────

class MN(nn.Module):
    """MobileNetV3 adapted for audio classification (from EfficientAT)."""

    def __init__(
        self,
        inverted_residual_setting: List[InvertedResidualConfig],
        last_channel: int,
        num_classes: int = 1000,
        block: Optional[Callable[..., nn.Module]] = None,
        norm_layer: Optional[Callable[..., nn.Module]] = None,
        dropout: float = 0.2,
        in_conv_kernel: int = 3,
        in_conv_stride: int = 2,
        in_channels: int = 1,
        **kwargs: Any,
    ) -> None:
        super().__init__()

        if not inverted_residual_setting:
            raise ValueError("The inverted_residual_setting should not be empty")
        elif not (
            isinstance(inverted_residual_setting, Sequence)
            and all(isinstance(s, InvertedResidualConfig) for s in inverted_residual_setting)
        ):
            raise TypeError("The inverted_residual_setting should be List[InvertedResidualConfig]")

        if block is None:
            block = InvertedResidual

        depthwise_norm_layer = norm_layer = (
            norm_layer if norm_layer is not None
            else partial(nn.BatchNorm2d, eps=0.001, momentum=0.01)
        )

        layers: List[nn.Module] = []

        # building first layer
        firstconv_output_channels = inverted_residual_setting[0].input_channels
        layers.append(
            ConvNormActivation(
                in_channels,
                firstconv_output_channels,
                kernel_size=in_conv_kernel,
                stride=in_conv_stride,
                norm_layer=norm_layer,
                activation_layer=nn.Hardswish,
            )
        )

        # get squeeze excitation config
        se_cnf = kwargs.get("se_conf", None)

        # track frequency and time dimensions through the network
        f_dim, t_dim = kwargs.get("input_dims", (128, 1000))
        f_dim = cnn_out_size(f_dim, 1, 1, 3, 2)
        t_dim = cnn_out_size(t_dim, 1, 1, 3, 2)
        for cnf in inverted_residual_setting:
            f_dim = cnf.out_size(f_dim)
            t_dim = cnf.out_size(t_dim)
            cnf.f_dim, cnf.t_dim = f_dim, t_dim
            layers.append(block(cnf, se_cnf, norm_layer, depthwise_norm_layer))

        # building last several layers
        lastconv_input_channels = inverted_residual_setting[-1].out_channels
        lastconv_output_channels = 6 * lastconv_input_channels
        layers.append(
            ConvNormActivation(
                lastconv_input_channels,
                lastconv_output_channels,
                kernel_size=1,
                norm_layer=norm_layer,
                activation_layer=nn.Hardswish,
            )
        )

        self.features = nn.Sequential(*layers)
        self.head_type = kwargs.get("head_type", False)
        if self.head_type == "multihead_attention_pooling":
            self.classifier = MultiHeadAttentionPooling(
                lastconv_output_channels,
                num_classes,
                num_heads=kwargs.get("multihead_attention_heads"),
            )
        elif self.head_type == "fully_convolutional":
            self.classifier = nn.Sequential(
                nn.Conv2d(lastconv_output_channels, num_classes, kernel_size=(1, 1), bias=False),
                nn.BatchNorm2d(num_classes),
                nn.AdaptiveAvgPool2d((1, 1)),
            )
        elif self.head_type == "mlp":
            self.classifier = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(start_dim=1),
                nn.Linear(lastconv_output_channels, last_channel),
                nn.Hardswish(inplace=True),
                nn.Dropout(p=dropout, inplace=True),
                nn.Linear(last_channel, num_classes),
            )
        else:
            raise NotImplementedError(
                f"Head '{self.head_type}' unknown. Must be one of: "
                "'mlp', 'fully_convolutional', 'multihead_attention_pooling'"
            )

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _forward_impl(
        self, x: Tensor, return_fmaps: bool = False
    ) -> Union[Tuple[Tensor, Tensor], Tuple[Tensor, List[Tensor]]]:
        fmaps = []
        for layer in self.features:
            x = layer(x)
            if return_fmaps:
                fmaps.append(x)

        features = F.adaptive_avg_pool2d(x, (1, 1)).squeeze()
        x = self.classifier(x).squeeze()

        if features.dim() == 1 and x.dim() == 1:
            # squeezed batch dimension
            features = features.unsqueeze(0)
            x = x.unsqueeze(0)

        if return_fmaps:
            return x, fmaps
        else:
            return x, features

    def forward(
        self, x: Tensor
    ) -> Union[Tuple[Tensor, Tensor], Tuple[Tensor, List[Tensor]]]:
        return self._forward_impl(x)


def _mobilenet_v3_conf(
    width_mult: float = 1.0,
    reduced_tail: bool = False,
    dilated: bool = False,
    strides: Tuple[int, ...] = (2, 2, 2, 2),
    **kwargs: Any,
) -> Tuple[List[InvertedResidualConfig], int]:
    reduce_divider = 2 if reduced_tail else 1
    dilation = 2 if dilated else 1

    bneck_conf = partial(InvertedResidualConfig, width_mult=width_mult)
    adjust_channels = partial(InvertedResidualConfig.adjust_channels, width_mult=width_mult)

    inverted_residual_setting = [
        bneck_conf(16, 3, 16, 16, False, "RE", 1, 1),
        bneck_conf(16, 3, 64, 24, False, "RE", strides[0], 1),
        bneck_conf(24, 3, 72, 24, False, "RE", 1, 1),
        bneck_conf(24, 5, 72, 40, True, "RE", strides[1], 1),
        bneck_conf(40, 5, 120, 40, True, "RE", 1, 1),
        bneck_conf(40, 5, 120, 40, True, "RE", 1, 1),
        bneck_conf(40, 3, 240, 80, False, "HS", strides[2], 1),
        bneck_conf(80, 3, 200, 80, False, "HS", 1, 1),
        bneck_conf(80, 3, 184, 80, False, "HS", 1, 1),
        bneck_conf(80, 3, 184, 80, False, "HS", 1, 1),
        bneck_conf(80, 3, 480, 112, True, "HS", 1, 1),
        bneck_conf(112, 3, 672, 112, True, "HS", 1, 1),
        bneck_conf(112, 5, 672, 160 // reduce_divider, True, "HS", strides[3], dilation),
        bneck_conf(160 // reduce_divider, 5, 960 // reduce_divider, 160 // reduce_divider, True, "HS", 1, dilation),
        bneck_conf(160 // reduce_divider, 5, 960 // reduce_divider, 160 // reduce_divider, True, "HS", 1, dilation),
    ]
    last_channel = adjust_channels(1280 // reduce_divider)
    return inverted_residual_setting, last_channel


def _mobilenet_v3(
    inverted_residual_setting: List[InvertedResidualConfig],
    last_channel: int,
    **kwargs: Any,
) -> MN:
    return MN(inverted_residual_setting, last_channel, **kwargs)


def get_model(
    num_classes: int = 527,
    pretrained_name: Optional[str] = None,
    width_mult: float = 1.0,
    reduced_tail: bool = False,
    dilated: bool = False,
    strides: Tuple[int, int, int, int] = (2, 2, 2, 2),
    head_type: str = "mlp",
    multihead_attention_heads: int = 4,
    input_dim_f: int = 128,
    input_dim_t: int = 1000,
    se_dims: str = "c",
    se_agg: str = "max",
    se_r: int = 4,
) -> MN:
    """Construct an EfficientAT MobileNetV3 model."""
    dim_map = {"c": 1, "f": 2, "t": 3}
    assert len(se_dims) <= 3 and all(s in dim_map for s in se_dims) or se_dims == "none"

    input_dims = (input_dim_f, input_dim_t)
    if se_dims == "none":
        se_dims_list = None
    else:
        se_dims_list = [dim_map[s] for s in se_dims]
    se_conf = dict(se_dims=se_dims_list, se_agg=se_agg, se_r=se_r)

    inverted_residual_setting, last_channel = _mobilenet_v3_conf(
        width_mult=width_mult,
        reduced_tail=reduced_tail,
        dilated=dilated,
        strides=strides,
    )
    return _mobilenet_v3(
        inverted_residual_setting,
        last_channel,
        num_classes=num_classes,
        head_type=head_type,
        multihead_attention_heads=multihead_attention_heads,
        input_dims=input_dims,
        se_conf=se_conf,
    )


print("Model classes defined: MN, get_model, InvertedResidual, InvertedResidualConfig,")
print("  ConcurrentSEBlock, SqueezeExcitation, MultiHeadAttentionPooling")

## Cell 6: AugmentMelSTFT preprocessing

**This is the most critical cell.** Exact copy of `src/acoustic/classification/efficientat/preprocess.py`. Every detail — conv1d preemphasis, STFT parameters, Kaldi mel banks, log normalization formula, SpecAugment — must be preserved exactly as written. Any deviation here will cause a train/inference mismatch.

In [ ]:
import torchaudio

# SOURCE: src/acoustic/classification/efficientat/preprocess.py
# DO NOT modify this class. Any change here breaks compatibility with local inference.


class AugmentMelSTFT(nn.Module):
    """Mel spectrogram with preemphasis and (log_mel + 4.5) / 5.0 normalization.

    Matches EfficientAT's preprocessing exactly for pretrained weight compatibility.
    Default params: 32kHz, 128 mels, win=800, hop=320, n_fft=1024.

    Pipeline:
      1. Preemphasis: conv1d with kernel [-0.97, 1]
      2. STFT: n_fft=1024, hop=320, win=800, hann (periodic=False), center=True
      3. Power spectrum: real^2 + imag^2
      4. Mel filterbank: torchaudio.compliance.kaldi.get_mel_banks()
      5. Log normalization: (log(mel + 1e-5) + 4.5) / 5.0
      6. SpecAugment: FrequencyMasking + TimeMasking during training only
    """

    def __init__(
        self,
        n_mels: int = 128,
        sr: int = 32000,
        win_length: int = 800,
        hopsize: int = 320,
        n_fft: int = 1024,
        freqm: int = 48,
        timem: int = 192,
        fmin: float = 0.0,
        fmax: float = None,
        fmin_aug_range: int = 10,
        fmax_aug_range: int = 2000,
    ) -> None:
        super().__init__()
        self.win_length = win_length
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.sr = sr
        self.fmin = fmin
        if fmax is None:
            fmax = sr // 2 - fmax_aug_range // 2
        self.fmax = fmax
        self.hopsize = hopsize
        self.register_buffer(
            "window", torch.hann_window(win_length, periodic=False), persistent=False
        )
        assert fmin_aug_range >= 1, f"fmin_aug_range={fmin_aug_range} should be >=1"
        assert fmax_aug_range >= 1, f"fmax_aug_range={fmax_aug_range} should be >=1"
        self.fmin_aug_range = fmin_aug_range
        self.fmax_aug_range = fmax_aug_range

        self.register_buffer(
            "preemphasis_coefficient", torch.as_tensor([[[-.97, 1]]]), persistent=False
        )
        if freqm == 0:
            self.freqm = torch.nn.Identity()
        else:
            self.freqm = torchaudio.transforms.FrequencyMasking(freqm, iid_masks=True)
        if timem == 0:
            self.timem = torch.nn.Identity()
        else:
            self.timem = torchaudio.transforms.TimeMasking(timem, iid_masks=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Convert waveform (batch, samples) to normalized mel spectrogram."""
        x = nn.functional.conv1d(x.unsqueeze(1), self.preemphasis_coefficient).squeeze(1)
        x = torch.stft(
            x,
            self.n_fft,
            hop_length=self.hopsize,
            win_length=self.win_length,
            center=True,
            normalized=False,
            window=self.window,
            return_complex=False,
        )
        x = (x ** 2).sum(dim=-1)  # power magnitude spectrum

        fmin = self.fmin + torch.randint(self.fmin_aug_range, (1,)).item()
        fmax = self.fmax + self.fmax_aug_range // 2 - torch.randint(self.fmax_aug_range, (1,)).item()
        # don't augment eval data
        if not self.training:
            fmin = self.fmin
            fmax = self.fmax

        mel_basis, _ = torchaudio.compliance.kaldi.get_mel_banks(
            self.n_mels, self.n_fft, self.sr,
            fmin, fmax, vtln_low=100.0, vtln_high=-500.0, vtln_warp_factor=1.0,
        )
        mel_basis = torch.as_tensor(
            torch.nn.functional.pad(mel_basis, (0, 1), mode="constant", value=0),
            device=x.device,
        )
        with torch.cuda.amp.autocast(enabled=False):
            melspec = torch.matmul(mel_basis, x)

        melspec = (melspec + 0.00001).log()

        if self.training:
            melspec = self.freqm(melspec)
            melspec = self.timem(melspec)

        melspec = (melspec + 4.5) / 5.0  # fast normalization
        return melspec


print("AugmentMelSTFT defined (exact match to local inference preprocess.py)")

## Cell 7: Focal Loss

In [ ]:
from torchvision.ops import sigmoid_focal_loss


class FocalLoss(nn.Module):
    """Focal loss for binary classification.

    Reduces easy-negative dominance compared to BCE.
    Uses torchvision.ops.sigmoid_focal_loss internally.
    """

    def __init__(self, alpha: float = 0.25, gamma: float = 2.0) -> None:
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return sigmoid_focal_loss(
            inputs, targets,
            alpha=self.alpha,
            gamma=self.gamma,
            reduction="mean",
        )


print(f"FocalLoss defined (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA})")

## Cell 8: Load DADS dataset from HuggingFace

Loads `geronimobasso/drone-audio-detection-samples`, decodes audio lazily (bytes only), and splits 80/20.

In [ ]:
from datasets import Audio, load_dataset

# Load with decode=False to get raw bytes — we decode ourselves with soundfile
# to maintain full control over resampling and avoid HF audio pipeline differences
# Using token= for authenticated download (faster, avoids rate limits)
ds = load_dataset(HF_DATASET, split="train", token=True)
ds = ds.cast_column("audio", Audio(decode=False))

# 80/20 train/val split — seed is fixed to ensure reproducibility
split = ds.train_test_split(test_size=0.2, seed=SEED)
train_ds = split["train"]
val_ds = split["test"]

# Class distribution
train_labels = train_ds["label"]
val_labels = val_ds["label"]
n_drone_train = sum(train_labels)
n_bg_train = len(train_labels) - n_drone_train
n_drone_val = sum(val_labels)
n_bg_val = len(val_labels) - n_drone_val

print(f"Dataset: {HF_DATASET}")
print(f"Total samples: {len(ds)}")
print(f"Train: {len(train_ds)} samples  (drone={n_drone_train}, background={n_bg_train})")
print(f"Val:   {len(val_ds)} samples  (drone={n_drone_val}, background={n_bg_val})")
print(f"Class balance (train): drone={n_drone_train/len(train_ds)*100:.1f}%  bg={n_bg_train/len(train_ds)*100:.1f}%")


## Cell 9: Dataset class

PyTorch Dataset that decodes WAV bytes, resamples 16kHz → 32kHz, and extracts random 1-second segments.

In [ ]:
import io
import random
from torch.utils.data import Dataset

import soundfile as sf


def decode_wav_bytes(wav_bytes: bytes) -> "np.ndarray":
    """Decode raw WAV bytes to a float32 mono numpy array.

    Mixes down to mono if multichannel. Returns float32 samples.
    """
    import numpy as np
    audio, sr = sf.read(io.BytesIO(wav_bytes), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio, sr


class DroneAudioDataset(Dataset):
    """PyTorch Dataset wrapping a HuggingFace drone audio split.

    Each item:
      - Reads WAV bytes from the HF dataset row
      - Decodes with soundfile (via io.BytesIO)
      - Resamples from SOURCE_SAMPLE_RATE (16kHz) to SAMPLE_RATE (32kHz)
      - Extracts a random 1-second (SEGMENT_SAMPLES) crop, with zero-padding if short
      - Returns (waveform: FloatTensor[SEGMENT_SAMPLES], label: FloatTensor[1])
    """

    def __init__(
        self,
        hf_dataset,
        segment_samples: int = SEGMENT_SAMPLES,
        source_sr: int = SOURCE_SAMPLE_RATE,
        target_sr: int = SAMPLE_RATE,
    ) -> None:
        self.ds = hf_dataset
        self.segment_samples = segment_samples
        self.source_sr = source_sr
        self.target_sr = target_sr

    def __len__(self) -> int:
        return len(self.ds)

    def __getitem__(self, idx: int):
        row = self.ds[idx]
        wav_bytes = row["audio"]["bytes"]
        label = row["label"]

        # Decode bytes → float32 numpy array
        audio, sr = decode_wav_bytes(wav_bytes)

        # Convert to tensor for torchaudio resampling
        waveform = torch.from_numpy(audio).unsqueeze(0)  # (1, T)

        # Resample if needed (DADS is 16kHz, model expects 32kHz)
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=self.target_sr)

        waveform = waveform.squeeze(0)  # (T,)

        # Random 1-second crop, or zero-pad if too short
        n = waveform.shape[0]
        if n >= self.segment_samples:
            start = random.randint(0, n - self.segment_samples)
            waveform = waveform[start : start + self.segment_samples]
        else:
            pad = self.segment_samples - n
            waveform = torch.nn.functional.pad(waveform, (0, pad))

        label_tensor = torch.tensor(float(label))
        return waveform, label_tensor


# Quick sanity check
_check_ds = DroneAudioDataset(train_ds)
_wav, _lbl = _check_ds[0]
print(f"Waveform shape: {_wav.shape}  dtype: {_wav.dtype}")
print(f"Label: {_lbl}  dtype: {_lbl.dtype}")
print(f"Waveform range: [{_wav.min():.4f}, {_wav.max():.4f}]")

## Cell 10: DataLoaders with WeightedRandomSampler

`WeightedRandomSampler` ensures each training batch sees roughly equal drone/background samples regardless of the dataset's natural class distribution.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler

torch.manual_seed(SEED)

# Build PyTorch datasets
train_dataset = DroneAudioDataset(train_ds)
val_dataset = DroneAudioDataset(val_ds)

# WeightedRandomSampler: inverse-frequency weights per class
labels_array = np.array(train_ds["label"])
class_counts = np.bincount(labels_array)          # [n_background, n_drone]
class_weights = 1.0 / class_counts.astype(float)  # higher weight for minority class
sample_weights = class_weights[labels_array]       # per-sample weight

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(train_dataset),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Train loader: {len(train_loader)} batches x {BATCH_SIZE} = {len(train_loader)*BATCH_SIZE} samples/epoch")
print(f"Val loader:   {len(val_loader)} batches x {BATCH_SIZE}")
print(f"Class weights: background={class_weights[0]:.4f}, drone={class_weights[1]:.4f}")

## Cell 11: Model setup

Build MN10 with 527-class head, load AudioSet pretrained weights, then swap the final linear layer to a binary (1-output) head.

In [ ]:
# Build model with 527-class head matching pretrained checkpoint
model = get_model(
    num_classes=NUM_CLASSES_PRETRAINED,
    width_mult=WIDTH_MULT,
    head_type="mlp",
    input_dim_f=INPUT_DIM_F,
    input_dim_t=INPUT_DIM_T,
)

# Load pretrained AudioSet weights
state_dict = torch.load(PRETRAINED_PATH, map_location="cpu")
model.load_state_dict(state_dict, strict=True)
print(f"Loaded pretrained weights from {PRETRAINED_PATH}")

# Replace the final classifier layer (classifier[-1]) with binary head
# classifier is: [AdaptiveAvgPool2d, Flatten, Linear(960,1280), Hardswish, Dropout, Linear(1280,527)]
#                                                                                              ^^ replace this
in_features = model.classifier[-1].in_features
print(f"Original head: Linear({in_features}, {model.classifier[-1].out_features})")
model.classifier[-1] = nn.Linear(in_features, 1)
print(f"Replaced head: Linear({in_features}, 1)")

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Cell 12: 3-stage transfer learning

**Stage 1** (10 epochs, lr=1e-3): Head only — freeze backbone, train only `classifier`.  
**Stage 2** (15 epochs, lr=1e-4): Head + last 3 feature blocks — `features[-3:]` unfrozen, CosineAnnealingLR.  
**Stage 3** (20 epochs, lr=1e-5): All layers — full fine-tune, CosineAnnealingLR.

Early stopping (patience=5) per stage. Best checkpoint per stage saved to disk.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix


# ── Preprocessing instances ───────────────────────────────────────────────────
# mel_train: SpecAugment ON  (freqm=48, timem=192)
# mel_eval:  SpecAugment OFF (freqm=0,  timem=0)
mel_train = AugmentMelSTFT(
    n_mels=N_MELS, sr=SAMPLE_RATE, win_length=WIN_LENGTH,
    hopsize=HOP_LENGTH, n_fft=N_FFT,
    freqm=FREQM_TRAIN, timem=TIMEM_TRAIN,
    fmin=FMIN, fmax=FMAX,
    fmin_aug_range=1,   # disable fmin augmentation (deterministic fmin=FMIN)
    fmax_aug_range=1,   # disable fmax augmentation (deterministic fmax=FMAX)
).to(DEVICE)

mel_eval = AugmentMelSTFT(
    n_mels=N_MELS, sr=SAMPLE_RATE, win_length=WIN_LENGTH,
    hopsize=HOP_LENGTH, n_fft=N_FFT,
    freqm=0, timem=0,
    fmin=FMIN, fmax=FMAX,
    fmin_aug_range=1,
    fmax_aug_range=1,
).to(DEVICE)

criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)


# ── Helper: one train epoch ───────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, mel_preprocess):
    model.train()
    mel_preprocess.train()
    total_loss = 0.0
    for wavs, labels in loader:
        wavs = wavs.to(DEVICE)            # (B, 32000)
        labels = labels.to(DEVICE)        # (B,)

        # Mel spectrogram: (B, 32000) → (B, 128, T) → (B, 1, 128, T)
        with torch.no_grad():
            specs = mel_preprocess(wavs)  # AugmentMelSTFT handles .training flag
        # Re-enable grad tracking for the model pass
        specs = specs.unsqueeze(1)        # add channel dim

        optimizer.zero_grad()
        logits, _ = model(specs)          # (B,)
        logits = logits.squeeze(-1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


# ── Helper: validation epoch ──────────────────────────────────────────────────
def val_epoch(model, loader, mel_preprocess):
    model.eval()
    mel_preprocess.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for wavs, labels in loader:
            wavs = wavs.to(DEVICE)
            labels = labels.to(DEVICE)

            specs = mel_preprocess(wavs).unsqueeze(1)
            logits, _ = model(specs)
            logits = logits.squeeze(-1)

            loss = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.long().cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
    return avg_loss, acc, prec, rec, f1, cm


# ── Stage runner ──────────────────────────────────────────────────────────────
def run_stage(stage_num, n_epochs, lr, scheduler_cls=None):
    """Run one training stage with early stopping."""
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
    )
    scheduler = None
    if scheduler_cls is not None:
        scheduler = scheduler_cls(optimizer, T_max=n_epochs)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_state = None

    print(f"\n{'='*70}")
    print(f"Stage {stage_num} | lr={lr}  epochs={n_epochs}  patience={EARLY_STOP_PATIENCE}")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")
    print(f"{'='*70}")

    for epoch in range(1, n_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, mel_train)
        val_loss, val_acc, prec, rec, f1, cm = val_epoch(model, val_loader, mel_eval)

        if scheduler is not None:
            scheduler.step()

        print(
            f"Stage {stage_num} | Epoch {epoch:3d}/{n_epochs} | "
            f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
            f"acc={val_acc:.4f}  prec={prec:.4f}  rec={rec:.4f}  f1={f1:.4f}"
        )
        print(f"  Confusion matrix (rows=true, cols=pred) [bg, drone]:\n  {cm}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            torch.save(best_state, f"stage{stage_num}_best.pt")
            print(f"  -> New best val_loss={best_val_loss:.4f} saved")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs)")
                break

    # Restore best weights before next stage
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Stage {stage_num} complete. Best val_loss={best_val_loss:.4f}")


# ── Stage 1: Head only ────────────────────────────────────────────────────────
print("Stage 1: Freezing backbone, training classifier head only")
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

run_stage(stage_num=1, n_epochs=STAGE1_EPOCHS, lr=LR_STAGE1, scheduler_cls=None)


# ── Stage 2: Head + last 3 feature blocks ────────────────────────────────────
print("\nStage 2: Unfreezing features[-3:] in addition to classifier")
for param in model.features[-3:].parameters():
    param.requires_grad = True

run_stage(
    stage_num=2,
    n_epochs=STAGE2_EPOCHS,
    lr=LR_STAGE2,
    scheduler_cls=lambda opt, T_max: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=T_max),
)


# ── Stage 3: All layers ───────────────────────────────────────────────────────
print("\nStage 3: Unfreezing all layers")
for param in model.parameters():
    param.requires_grad = True

run_stage(
    stage_num=3,
    n_epochs=STAGE3_EPOCHS,
    lr=LR_STAGE3,
    scheduler_cls=lambda opt, T_max: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=T_max),
)

print("\nTraining complete.")

## Cell 13: Save, verify, and print final metrics

Saves `model.state_dict()` (NOT full model). Verifies by loading into a fresh MN10 instance and running a forward pass. This confirms the saved weights are loadable with the same architecture code used locally.

In [ ]:
# ── Save state_dict ───────────────────────────────────────────────────────────
torch.save(model.state_dict(), CHECKPOINT_PATH)
size_mb = os.path.getsize(CHECKPOINT_PATH) / 1e6
print(f"Saved: {CHECKPOINT_PATH}  ({size_mb:.1f} MB)")

# ── Verify: load into fresh model and run inference ───────────────────────────
print("\n--- Verify Compatibility ---")
verify_model = get_model(
    num_classes=NUM_CLASSES_PRETRAINED,
    width_mult=WIDTH_MULT,
    head_type="mlp",
    input_dim_f=INPUT_DIM_F,
    input_dim_t=INPUT_DIM_T,
)
# Replace head to match binary architecture
verify_model.classifier[-1] = nn.Linear(in_features, 1)

# Load saved state dict
saved_state = torch.load(CHECKPOINT_PATH, map_location="cpu")
verify_model.load_state_dict(saved_state, strict=True)
verify_model = verify_model.to(DEVICE)
verify_model.eval()
print("State dict loaded successfully into fresh model instance")

# Run a few inference samples from the val set
mel_verify = AugmentMelSTFT(
    n_mels=N_MELS, sr=SAMPLE_RATE, win_length=WIN_LENGTH,
    hopsize=HOP_LENGTH, n_fft=N_FFT,
    freqm=0, timem=0,       # eval mode: no augmentation
    fmin=FMIN, fmax=FMAX,
    fmin_aug_range=1,
    fmax_aug_range=1,
).to(DEVICE)
mel_verify.eval()

print("\nSample inference (5 val items):")
print(f"{'idx':>4}  {'label':>6}  {'prob':>6}  {'pred':>6}  {'correct':>7}")
print("-" * 35)
for i in range(5):
    wav, label = val_dataset[i]
    wav_batch = wav.unsqueeze(0).to(DEVICE)     # (1, 32000)
    with torch.no_grad():
        spec = mel_verify(wav_batch).unsqueeze(1)  # (1, 1, 128, T)
        logit, _ = verify_model(spec)
        prob = torch.sigmoid(logit).item()
        pred = int(prob >= 0.5)
    correct = "YES" if pred == int(label.item()) else "NO"
    print(f"{i:>4}  {'drone' if int(label.item()) else 'bg':>6}  {prob:>6.3f}  {'drone' if pred else 'bg':>6}  {correct:>7}")

# ── Final metrics on full val set ─────────────────────────────────────────────
print("\n--- Final validation metrics ---")
final_loss, final_acc, final_prec, final_rec, final_f1, final_cm = val_epoch(
    verify_model, val_loader, mel_verify
)
print(f"Val loss:      {final_loss:.4f}")
print(f"Val accuracy:  {final_acc:.4f}  ({final_acc*100:.2f}%)")
print(f"Precision:     {final_prec:.4f}")
print(f"Recall:        {final_rec:.4f}")
print(f"F1:            {final_f1:.4f}")
print(f"\nConfusion matrix (rows=true [bg, drone], cols=pred):")
print(final_cm)

## Cell 14: Download checkpoint from Colab

In [ ]:
from google.colab import files

print(f"Downloading {CHECKPOINT_PATH} ...")
files.download(CHECKPOINT_PATH)
print("Done. Place the downloaded file at:")
print("  src/acoustic/classification/efficientat/weights/efficientat_mn10_local.pt")